# Milestone 3 Wallet Walkthrough

This notebook is a hands-on harness for the current Milestone 3 wallet surface.

It reuses the existing wallet backfill to:
- backfill a leaderboard-seeded wallet cohort into isolated per-run storage
- inspect raw wallet-universe and per-wallet capture paths
- visualize cohort-level profile comparisons
- drill into one wallet at a time with activity, closed-position, and open-position views
- keep partial endpoint failures visible instead of hiding them


## Workflow

Run the notebook top to bottom for a fresh session, or set `POLYMARKET_EXPLORATION_RUN_ID` before opening Jupyter to revisit an existing run directory.

The notebook keeps all artifacts under `data/runs/<run_id>/` so wallet exploration does not mix with the shared default warehouse.


In [ ]:
%matplotlib inline

import html
import importlib
import os
import sys
from datetime import UTC, datetime
from decimal import Decimal
from pathlib import Path
from pprint import pprint

import duckdb
import ipywidgets as widgets
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, clear_output, display
from dotenv import load_dotenv

REPO_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if all((candidate / marker).exists() for marker in ("PROJECT_SPEC.md", "ARCHITECTURE.md", "TASKS.md")) and (candidate / "src").is_dir():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root from the current working directory.")

bootstrap_spec = importlib.util.spec_from_file_location("notebook_bootstrap", REPO_ROOT / "notebook_bootstrap.py")
if bootstrap_spec is None or bootstrap_spec.loader is None:
    raise RuntimeError(f"Could not load notebook bootstrap helper from {REPO_ROOT / 'notebook_bootstrap.py'}.")
notebook_bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(notebook_bootstrap)
notebook_bootstrap.prepare_repo_imports(REPO_ROOT)

from src.ingestion import WalletUniverseSelectionRule, run_wallet_backfill
from src.research import (
    SUPPORTED_COHORT_METRICS,
    get_table_counts,
    latest_raw_capture_path,
    list_latest_wallet_dataset_captures,
    list_wallet_activity_trades,
    list_wallet_closed_position_points,
    list_wallet_cohort_profiles,
    list_wallet_open_positions,
    wallet_display_name,
)

load_dotenv(REPO_ROOT / ".env")

RUN_ID = os.environ.get(
    "POLYMARKET_EXPLORATION_RUN_ID",
    datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ"),
)
RUN_ROOT = REPO_ROOT / "data" / "runs" / RUN_ID
RAW_DIR = RUN_ROOT / "raw"
WAREHOUSE_PATH = RUN_ROOT / "warehouse" / "polymarket.duckdb"

RUN_ROOT.mkdir(parents=True, exist_ok=True)
WAREHOUSE_PATH.parent.mkdir(parents=True, exist_ok=True)

plt.style.use("ggplot")

COHORT_METRIC_LABELS = {
    "realized_pnl": "Realized PnL",
    "realized_roi": "Realized ROI",
    "hit_rate": "Hit Rate",
    "activity_volume_usdc": "Activity Volume (USDC)",
    "activity_trade_count": "Activity Trade Count",
    "closed_position_count": "Closed Position Count",
}


def decimal_to_float(value) -> float:
    if value is None:
        return 0.0
    if isinstance(value, Decimal):
        return float(value)
    return float(value)


def build_html_table(rows: list[dict[str, object]], columns: list[str]) -> str:
    if not rows:
        return "<p><em>No rows captured.</em></p>"
    header = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body_rows = []
    for row in rows:
        cells = "".join(f"<td>{html.escape(str(row.get(column, '')))}</td>" for column in columns)
        body_rows.append(f"<tr>{cells}</tr>")
    return (
        "<table>"
        "<thead><tr>" + header + "</tr></thead>"
        "<tbody>" + "".join(body_rows) + "</tbody>"
        "</table>"
    )


def normalize_value(value):
    if isinstance(value, Decimal):
        return f"{value:.4f}"
    if isinstance(value, datetime):
        return value.isoformat()
    return value


def profile_summary(profile) -> dict[str, object]:
    return {
        "wallet_address": profile.wallet_address,
        "rank": profile.rank,
        "user_name": profile.user_name,
        "verified_badge": profile.verified_badge,
        "leaderboard_pnl": normalize_value(profile.leaderboard_pnl),
        "leaderboard_volume": normalize_value(profile.leaderboard_volume),
        "realized_pnl": normalize_value(profile.realized_pnl),
        "realized_roi": normalize_value(profile.realized_roi),
        "hit_rate": normalize_value(profile.hit_rate),
        "activity_trade_count": profile.activity_trade_count,
        "activity_volume_usdc": normalize_value(profile.activity_volume_usdc),
        "closed_position_count": profile.closed_position_count,
        "as_of_time_utc": normalize_value(profile.as_of_time_utc),
    }


print(f"REPO_ROOT={REPO_ROOT}")
print(f"RUN_ID={RUN_ID}")
print(f"RAW_DIR={RAW_DIR}")
print(f"WAREHOUSE_PATH={WAREHOUSE_PATH}")
print(f"SUPPORTED_COHORT_METRICS={SUPPORTED_COHORT_METRICS}")


In [ ]:
REQUIRED_MODULES = ("duckdb", "matplotlib", "ipywidgets", "notebook")
for module_name in REQUIRED_MODULES:
    module = importlib.import_module(module_name)
    print(f"{module_name}: {getattr(module, '__version__', 'version unavailable')}")

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")


## 1. Wallet Backfill Parameters

Adjust these limits before running the next cell if you want a smaller or larger cohort snapshot.


In [ ]:
LEADERBOARD_LIMIT = 10
POSITIONS_LIMIT = 100
CLOSED_POSITIONS_LIMIT = 100
ACTIVITY_LIMIT = 100

print(
    {
        "leaderboard_limit": LEADERBOARD_LIMIT,
        "positions_limit": POSITIONS_LIMIT,
        "closed_positions_limit": CLOSED_POSITIONS_LIMIT,
        "activity_limit": ACTIVITY_LIMIT,
    }
)


## 2. Backfill The Wallet Cohort

This cell calls `run_wallet_backfill(...)` directly so the notebook can keep partial failures visible while still rendering the rest of the walkthrough.


In [ ]:
summary = run_wallet_backfill(
    raw_data_dir=RAW_DIR,
    warehouse_path=WAREHOUSE_PATH,
    selection_rule=WalletUniverseSelectionRule(leaderboard_limit=LEADERBOARD_LIMIT),
    positions_limit=POSITIONS_LIMIT,
    closed_positions_limit=CLOSED_POSITIONS_LIMIT,
    activity_limit=ACTIVITY_LIMIT,
)

print("Wallet backfill summary")
pprint(
    {
        "selection_rule": summary.selection_rule,
        "seed_wallet_addresses": summary.seed_wallet_addresses,
        "position_rows": summary.position_rows,
        "closed_position_rows": summary.closed_position_rows,
        "activity_trade_rows": summary.activity_trade_rows,
        "wallet_profile_rows": summary.wallet_profile_rows,
        "raw_capture_count": summary.raw_capture_count,
    }
)

if summary.skipped_wallets:
    print("Skipped wallet seeds:")
    for skipped_wallet in summary.skipped_wallets:
        print(f"- identifier={skipped_wallet.identifier} reason={skipped_wallet.reason}")

if summary.failures:
    print("Partial failures were captured during this run:")
    for failure in summary.failures:
        print(
            f"- wallet_address={failure.wallet_address} dataset={failure.dataset} detail={failure.detail}"
        )
else:
    print("No partial failures were captured during this run.")


## 3. Quality Checks And Raw Capture Inspection

Inspect the normalized wallet tables and the latest wallet-universe raw captures before plotting.


In [ ]:
table_counts = get_table_counts(
    WAREHOUSE_PATH,
    table_names=("wallet_profiles", "wallet_positions", "wallet_closed_positions", "trades"),
)
pprint(table_counts)

latest_raw_paths = {
    "wallet_universe_leaderboard": latest_raw_capture_path(RAW_DIR, "data_api", "wallet_universe_leaderboard"),
    "wallet_seed_list": latest_raw_capture_path(RAW_DIR, "data_api", "wallet_seed_list"),
}
pprint(latest_raw_paths)

for dataset in ("wallet_positions", "wallet_closed_positions", "wallet_activity"):
    print(f"\nLatest per-wallet raw captures for {dataset}:")
    capture_rows = [
        {
            "wallet_address": capture.wallet_address,
            "record_count": capture.record_count,
            "collection_time_utc": capture.collection_time_utc,
            "path": str(capture.path),
        }
        for capture in list_latest_wallet_dataset_captures(RAW_DIR, dataset)
    ]
    if not capture_rows:
        print("<no raw captures>")
    else:
        pprint(capture_rows)


## 4. Cohort Overview

Start with the selected wallet universe, then compare the profile cohort with a ranking control and a wallet-comparison scatter plot.


In [ ]:
COHORT_ROWS = list_wallet_cohort_profiles(WAREHOUSE_PATH, RAW_DIR)
if not COHORT_ROWS:
    raise RuntimeError("Wallet backfill completed without writing any wallet profile rows.")

COHORT_INDEX = {row.wallet_address: row for row in COHORT_ROWS}
WALLET_LABELS = {
    row.wallet_address: wallet_display_name(row.wallet_address, user_name=row.user_name)
    for row in COHORT_ROWS
}

print(f"Wallet cohort rows: {len(COHORT_ROWS)}")
for row in COHORT_ROWS[: min(5, len(COHORT_ROWS))]:
    pprint(profile_summary(row))

leaderboard_rows = [row for row in COHORT_ROWS if row.leaderboard_pnl is not None]
if leaderboard_rows:
    fig, ax = plt.subplots(figsize=(12, 5))
    labels = [f"#{row.rank or '?'} {WALLET_LABELS[row.wallet_address]}" for row in leaderboard_rows]
    values = [decimal_to_float(row.leaderboard_pnl) for row in leaderboard_rows]
    ax.bar(labels, values, color="tab:blue")
    ax.set_title("Selected wallet seeds by leaderboard PnL")
    ax.set_ylabel("Leaderboard PnL")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

metric_dropdown = widgets.Dropdown(
    options=[(COHORT_METRIC_LABELS[metric], metric) for metric in SUPPORTED_COHORT_METRICS],
    value="realized_pnl",
    description="Metric",
)
top_n_slider = widgets.IntSlider(
    value=min(5, len(COHORT_ROWS)),
    min=1,
    max=max(1, len(COHORT_ROWS)),
    step=1,
    description="Top N",
)
cohort_metric_output = widgets.Output()


def render_metric_chart(*_args) -> None:
    with cohort_metric_output:
        clear_output(wait=True)
        metric = metric_dropdown.value
        top_n = top_n_slider.value
        metric_rows = [row for row in COHORT_ROWS if getattr(row, metric) is not None]
        metric_rows.sort(key=lambda row: decimal_to_float(getattr(row, metric)), reverse=True)
        if not metric_rows:
            print(f"No rows have a non-null value for {metric}.")
            return
        selected_rows = metric_rows[:top_n]
        fig, ax = plt.subplots(figsize=(12, 5))
        labels = [WALLET_LABELS[row.wallet_address] for row in selected_rows]
        values = [decimal_to_float(getattr(row, metric)) for row in selected_rows]
        ax.bar(labels, values, color="tab:green")
        ax.set_title(f"Top wallets by {COHORT_METRIC_LABELS[metric]}")
        ax.set_ylabel(COHORT_METRIC_LABELS[metric])
        ax.tick_params(axis="x", rotation=30)
        plt.tight_layout()
        plt.show()


metric_dropdown.observe(render_metric_chart, names="value")
top_n_slider.observe(render_metric_chart, names="value")
render_metric_chart()
display(widgets.HBox([metric_dropdown, top_n_slider]), cohort_metric_output)

fig, ax = plt.subplots(figsize=(12, 6))
for row in COHORT_ROWS:
    x_value = decimal_to_float(row.activity_volume_usdc)
    y_value = decimal_to_float(row.realized_pnl)
    point_size = 120 + (row.closed_position_count * 50)
    point_color = "tab:blue" if row.verified_badge else "tab:orange"
    ax.scatter(x_value, y_value, s=point_size, alpha=0.8, color=point_color)
    ax.annotate(WALLET_LABELS[row.wallet_address], (x_value, y_value), fontsize=9, xytext=(5, 5), textcoords="offset points")
ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--")
ax.set_title("Wallet comparison: activity volume vs realized PnL")
ax.set_xlabel("Activity volume (USDC)")
ax.set_ylabel("Realized PnL")
plt.tight_layout()
plt.show()


## 5. Wallet Drilldown

Use the selector to inspect one wallet at a time. The plots show trade notional by side, cumulative closed-position realized PnL, and open-position current value, followed by recent-record tables.


In [ ]:
wallet_selector = widgets.Dropdown(
    options=[
        (f"{WALLET_LABELS[row.wallet_address]} [{row.wallet_address}]", row.wallet_address)
        for row in COHORT_ROWS
    ],
    value=COHORT_ROWS[0].wallet_address,
    description="Wallet",
    layout=widgets.Layout(width="80%"),
)
wallet_output = widgets.Output()


def render_wallet_drilldown(wallet_address: str) -> None:
    profile = COHORT_INDEX[wallet_address]
    activity_rows = list_wallet_activity_trades(WAREHOUSE_PATH, wallet_address)
    closed_points = list_wallet_closed_position_points(WAREHOUSE_PATH, wallet_address)
    open_positions = list_wallet_open_positions(WAREHOUSE_PATH, wallet_address)

    with wallet_output:
        clear_output(wait=True)
        display(Markdown(f"### {WALLET_LABELS[wallet_address]}"))
        pprint(profile_summary(profile))

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        if activity_rows:
            buy_rows = [row for row in activity_rows if (row.side or "").upper() == "BUY"]
            sell_rows = [row for row in activity_rows if (row.side or "").upper() == "SELL"]
            if buy_rows:
                axes[0].bar(
                    [row.trade_time_utc for row in buy_rows],
                    [decimal_to_float(row.volume_usdc) for row in buy_rows],
                    color="tab:green",
                    label="BUY",
                    alpha=0.8,
                )
            if sell_rows:
                axes[0].bar(
                    [row.trade_time_utc for row in sell_rows],
                    [-decimal_to_float(row.volume_usdc) for row in sell_rows],
                    color="tab:red",
                    label="SELL",
                    alpha=0.8,
                )
            axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
            axes[0].set_title("Activity notional by side")
            axes[0].set_ylabel("Notional (USDC)")
            axes[0].legend()
        else:
            axes[0].text(0.5, 0.5, "No activity trades captured", ha="center", va="center")
            axes[0].set_axis_off()

        if closed_points:
            axes[1].plot(
                [row.closed_at_utc for row in closed_points],
                [decimal_to_float(row.cumulative_realized_pnl) for row in closed_points],
                color="tab:blue",
                marker="o",
            )
            axes[1].axhline(0.0, color="black", linewidth=1.0, linestyle="--")
            axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
            axes[1].set_title("Cumulative closed-position realized PnL")
            axes[1].set_ylabel("PnL")
        else:
            axes[1].text(0.5, 0.5, "No closed positions captured", ha="center", va="center")
            axes[1].set_axis_off()

        if open_positions:
            labels = [row.label for row in open_positions]
            values = [decimal_to_float(row.current_value) for row in open_positions]
            axes[2].barh(labels, values, color="tab:purple")
            axes[2].set_title("Open-position current value")
            axes[2].set_xlabel("Current value")
        else:
            axes[2].text(0.5, 0.5, "No open positions captured", ha="center", va="center")
            axes[2].set_axis_off()

        plt.tight_layout()
        plt.show()

        display(Markdown("#### Recent activity trades"))
        display(
            HTML(
                build_html_table(
                    [
                        {
                            "trade_time_utc": normalize_value(row.trade_time_utc),
                            "side": row.side,
                            "label": row.label,
                            "volume_usdc": normalize_value(row.volume_usdc),
                            "transaction_hash": row.transaction_hash,
                        }
                        for row in activity_rows[:10]
                    ],
                    ["trade_time_utc", "side", "label", "volume_usdc", "transaction_hash"],
                )
            )
        )

        display(Markdown("#### Recent closed positions"))
        display(
            HTML(
                build_html_table(
                    [
                        {
                            "closed_at_utc": normalize_value(row.closed_at_utc),
                            "label": row.label,
                            "realized_pnl": normalize_value(row.realized_pnl),
                            "cumulative_realized_pnl": normalize_value(row.cumulative_realized_pnl),
                        }
                        for row in closed_points[-10:]
                    ],
                    ["closed_at_utc", "label", "realized_pnl", "cumulative_realized_pnl"],
                )
            )
        )

        display(Markdown("#### Open-position snapshot"))
        display(
            HTML(
                build_html_table(
                    [
                        {
                            "label": row.label,
                            "current_value": normalize_value(row.current_value),
                            "size": normalize_value(row.size),
                            "average_price": normalize_value(row.average_price),
                            "end_time_utc": normalize_value(row.end_time_utc),
                        }
                        for row in open_positions[:10]
                    ],
                    ["label", "current_value", "size", "average_price", "end_time_utc"],
                )
            )
        )


def handle_wallet_change(change) -> None:
    if change.get("name") == "value" and change.get("new"):
        render_wallet_drilldown(change["new"])


wallet_selector.observe(handle_wallet_change, names="value")
render_wallet_drilldown(wallet_selector.value)
display(wallet_selector, wallet_output)
